# **Regression: Ordinary Least Squares (OLS) Linear Regression**

## **Justification of Preprocessing Strategy**

### **Scale Invariance vs. Coefficient Interpretation**
Ordinary Least Squares (OLS) **Linear Regression** is mathematically scale-invariant regarding its final predictions. This means that whether the data is raw, standardized, or normalized, the model will output the exact same predictions. However, the magnitude and scale of the learned coefficients ($\beta$ weights) depend entirely on the scale of their corresponding features. To ensure that we can directly compare the clinical importance of each feature on the `diabetes_risk_score` (interpreting which variable has the strongest impact), we will evaluate both **Standardization** and **Normalization** to find the most numerically stable representation.

### **Data Integrity and Leakage Prevention**
To successfully shift our objective from classification to regression, we strictly drop the previous classification targets (`diagnosed_diabetes` and `diabetes_stage`) to avoid any data leakage. Furthermore, because the target variable `diabetes_risk_score` is continuous, we **omit the stratification parameter** during the train-test split, as stratification is mathematically exclusive to categorical classes.


## **Experiment Design**

We have designed a tournament of **2 focused runs** to establish our linear baseline environment, evaluating performance using **MAE, RMSE, and $R^2$**:

* **Standardized OLS Linear Regression**: Training the classical linear model on features processed via `StandardScaler` to evaluate performance under a normally distributed feature space.
* **Normalized OLS Linear Regression**: Training the classical linear model on features processed via `MinMaxScaler` to evaluate performance under a strictly bounded [0, 1] feature space.


In [3]:
import pandas as pd
import numpy as np
import time
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Regression_Linear_Regression")

<Experiment: artifact_location=('file:c:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente '
 'de '
 'Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/notebooks/Regression/LinearRegression/mlruns/14'), creation_time=1779132783652, experiment_id='14', last_update_time=1779132783652, lifecycle_stage='active', name='Regression_Linear_Regression', tags={}, workspace='default'>

In [ ]:
# Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")
categorical_cols = ['gender', 'ethnicity', 'smoking_status', 'education_level', 'employment_status', 'age_groups', 'weight_status', 'income_level']
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Drop classification targets to avoid data leakage
X = df_final.drop(["diabetes_risk_score", "diagnosed_diabetes", "diabetes_stage"], axis=1, errors='ignore')
y = df_final['diabetes_risk_score']

# Split data (80/20) - NOTE: Continuous target means NO stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

def log_regression_metrics(y_true, y_pred, duration):
    """Utility function to log regression evaluation metrics to MLflow"""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2_score", r2)
    mlflow.log_metric("fit_time", duration)

scalers = {
    "Standardization": StandardScaler(),
    "Normalization": MinMaxScaler()
}

# ---------------------------------------------------------
# 2 RUNS (Standardization vs Normalization)
# ---------------------------------------------------------
for s_name, scaler_obj in scalers.items():
    with mlflow.start_run(run_name=f"OLS_LinearReg_{s_name}"):
        # Create copies and apply scaling to numerical features
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()
        X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
        X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])
        
        # Initialize and train the pure OLS model
        model = LinearRegression()
        
        start_time = time.time()
        model.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        # Predict on testing set
        y_pred = model.predict(X_test_scaled)
        
        # Log parameters to MLflow
        mlflow.log_param("model_type", "OLS_LinearRegression")
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("fit_intercept", model.fit_intercept)
        
        # Log evaluation metrics
        log_regression_metrics(y_test, y_pred, duration)

## **Winner Run Selection (Priority Elimination Framework)**

### **Selection Criteria (in priority order)**
1. **Priority 1 (60% weight): Lowest MAE** — Clinical proximity; minimizes average day-to-day prediction error.
2. **Priority 2 (30% weight): RMSE proportional to MAE** — Rejects runs where RMSE spikes relative to MAE, indicating catastrophic errors.
3. **Priority 3 (10% weight): Acceptable R²** — Confirms statistical fit quality.
4. **Tiebreaker: Lowest Fit Time** — Applied only if a technical tie exists in MAE, RMSE, and R².

### **All Runs: Summary Table with Metrics**

| Run | Model | Scaler | MAE | RMSE | R² Score | Fit Time (s) | 
|---|---|---|---:|---:|---:|---:|
| **OLS_LinearReg_Standardization** | **OLS** | **Standardization** | **0.40386** | **0.71129** | **0.993869465** | **0.20186** |  
| OLS_LinearReg_Normalization | OLS | Normalization | 0.40386 | 0.71129 | 0.993869465 | 0.22208 | 


### **Step-by-Step Elimination Process**

**Step 1: Filter by Lowest MAE (Priority 1 — 60%)**
- Threshold: MAE ≤ 0.40386
- Candidates passing: OLS_LinearReg_Standardization (0.40386), OLS_LinearReg_Normalization (0.40386)
- Status: Both runs achieve identical MAE

**Step 2: Verify RMSE Proportional to MAE (Priority 2 — 30%)**
- Both candidates: RMSE = 0.71129 (identical)
- MAE-to-RMSE ratio: 0.40386 / 0.71129 ≈ 0.568 (same for both)
- Status: **No catastrophic divergence detected**. Both candidates pass.

**Step 3: Confirm Acceptable R² (Priority 3 — 10%)**
- Both candidates: R² = 0.993869465 (identical, excellent fit)
- Status: Both candidates confirmed acceptable.

**Step 4: Apply Tiebreaker — Lowest Fit Time**
- OLS_LinearReg_Standardization: 0.20186 s ← **LOWER** 
- OLS_LinearReg_Normalization: 0.22208 s


### **Final Decision**
**Winner: OLS_LinearReg_Standardization**

**Justification:** Both runs achieve identical predictive performance (MAE, RMSE, R²), as expected for OLS predictions which are scale-invariant. The tiebreaker rule selects OLS_LinearReg_Standardization due to its lower fit time (0.20186 s vs 0.22208 s). This demonstrates that while OLS predictions remain mathematically identical regardless of scaling, StandardScaler produces marginally faster training than MinMaxScaler in this implementation.